# **Training Model From Scratch**

In [ ]:
!nvidia-smi

In [ ]:
!unzip /content/archive.zip

In [ ]:
IMAGE_SIZE = [224, 224]
train_path = '/content/train'
valid_path = '/content/test'

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout, Input
from tensorflow.keras.applications.vgg16 import VGG16
from glob import glob
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img

In [ ]:
folders = glob('/content/train/*')
folders

In [ ]:
num_of_class = len(folders)
num_of_class

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/train',
    labels = "inferred",
    label_mode = "int",
    batch_size = 32,
    image_size = (256, 256)
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/test',
    labels = "inferred",
    label_mode = "int",
    batch_size = 32,
    image_size = (256, 256)
)

In [ ]:
def normailzation(image, label):
  image = tf.cast(image/255, tf.float32)
  return image, label

train_ds = train_ds.map(normailzation)
validation_ds = validation_ds.map(normailzation)

In [ ]:
model = Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(224, 224, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
training_set = train_datagen.flow_from_directory(
    '/content/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_set = test_datagen.flow_from_directory(
    '/content/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

In [ ]:
img_class = model.fit(training_set, epochs=50, validation_data=test_set, steps_per_epoch=len(training_set), validation_steps=len(test_set))

In [ ]:
model.evaluate(test_set)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.plot(img_class.history['loss'],'r',label='training loss')
plt.plot(img_class.history['val_loss'],label='validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(img_class.history['accuracy'],'r',label='training accuracy')
plt.plot(img_class.history['val_accuracy'],label='validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
import cv2

test_img = cv2.imread('/content/test/apples/img_p1_131.jpeg')
test_img

In [ ]:
plt.imshow(test_img)

In [ ]:
test_img.shape

In [ ]:
test_img = cv2.resize(test_img, (256, 256))
test_img.shape

In [ ]:
import cv2

resized_test_img = cv2.resize(test_img, (224, 224))
test_input = resized_test_img.reshape((1, 224, 224, 3))
result = model.predict(test_input)
result

In [ ]:
if int(result[0][0]) == 0:
  print("Apple")
else:
  print("Tomato")

# **In Pytorch Version**

In [ ]:
import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torchvision import datasets
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(
  root='/content/train',
  transform=image_transform,
  target_transform=None
)

validation_data = datasets.ImageFolder(
  root='/content/test',
  transform=image_transform,
  target_transform=None
)

In [ ]:
train_data = torch.utils.data.DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)

validation_data = torch.utils.data.DataLoader(
    dataset=validation_data,
    batch_size=32,
    shuffle=False
)

In [ ]:
class image_classifier_pytorch(nn.Module):
  def __init__(self, input_shape : int, hidden_units: int, output_shape: int):
    super().__init__()
    self.block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_units,     kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                  kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.block_2 = nn.Sequential(
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*64*64, out_features=output_shape)
    )

  def forward(self, x):
    x = self.block_1(x)
    x = self.block_2(x)
    x = self.classifier(x)
    return x

torch.manual_seed(42)
model0 = image_classifier_pytorch(input_shape=3, hidden_units=10, output_shape=1).to(device)
model0

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params=model0.parameters(), lr=0.001)

In [ ]:
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct/y_pred.numel())*100
  return acc

def train_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn, optimizer: torch.optim.Optimizer, device: torch.device):
  train_loss, train_acc = 0, 0
  model.to(device)
  for batch, (X, y) in enumerate(data_loader):
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred, y.unsqueeze(dim=1).type(torch.float))
    y_pred_class = (y_pred > 0).squeeze().long()
    train_acc += accuracy_fn(y, y_pred_class)
    train_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")

def test_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn, device: torch.device = device):
  test_loss, test_acc = 0, 0
  model.to(device)
  model.eval()

  with torch.inference_mode():
    for batch, (X, y) in enumerate(data_loader):
      X, y = X.to(device), y.to(device)
      test_pred = model(X)
      loss = loss_fn(test_pred, y.unsqueeze(dim=1).type(torch.float))
      test_pred_class = (test_pred > 0).squeeze().long()
      test_acc += accuracy_fn(y, test_pred_class)
      test_loss += loss.item()

  test_loss /= len(data_loader)
  test_acc /= len(data_loader)
  print(f"Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%")

In [ ]:
from functools import total_ordering
from timeit import default_timer as timer
torch.manual_seed(42)

train_time_start_on_cpu = timer()
epochs = 3

for epoch in range(epochs):
  print(f"Epoch: {epoch}\n---------")
  train_step(
      data_loader=train_data,
      model=model0,
      loss_fn=loss_fn,
      accuracy_fn=accuracy_fn,
      optimizer=optimizer,
      device=device
  )
  test_step(
      data_loader=validation_data,
      model=model0,
      loss_fn=loss_fn,
      accuracy_fn=accuracy_fn,
      device=device
  )

train_time_end_on_cpu = timer()
total_train_time_model_2 = train_time_end_on_cpu - train_time_start_on_cpu
print(f"Total training time: {total_train_time_model_2:.2f} seconds")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(img_class.history['loss'], label='Training Loss')
plt.plot(img_class.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(img_class.history['accuracy'], label='Training Accuracy')
plt.plot(img_class.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import cv2
import tensorflow as tf

def predict_fruit(image_path, model):
  img = cv2.imread(image_path)
  img = cv2.resize(img, (256, 256))
  img_input = tf.cast(img / 255.0, tf.float32)
  img_input = tf.reshape(img_input, (1, 256, 256, 3))
  prediction = model.predict(img_input)

  print(f"Raw model prediction: {prediction[0][0]:.4f}")

  class_names = ['apples', 'tomatoes']
  print(f"Class mapping: {class_names[0]} = 0, {class_names[1]} = 1")

  if prediction[0][0] > 0.5:
    return class_names[1].capitalize()
  else:
    return class_names[0].capitalize()

image_path_to_predict = '/content/test/apples/img_p1_131.jpeg'
predicted_class = predict_fruit(image_path_to_predict, model)
print(f"The image is predicted to be: {predicted_class}")

# **Data_Augmentation and Extained version**

In [ ]:
model1 = VGG16()
model1.summary()

In [ ]:
for i in range(len(model1.layers)):
  if 'conv' not in model1.layers[i].name:
    continue
  filters, bias = model1.layers[i].get_weights()
  print("Layer Number", i, model1.layers[i].name, filters.shape)

In [ ]:
filters, bias = model1.layers[1].get_weights()
filters.shape, bias.shape

In [ ]:
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)

In [ ]:
n_filters = 6
ix = 1
fig = plt.figure(figsize=(15, 10))
for i in range(n_filters):
  f = filters[:, :, :, i]
  for j in range(3):
    plt.subplot(n_filters,3,ix)
    plt.imshow(f[:,:,j] ,cmap='gray')
    ix+=1
plt.show()

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img
from keras.applications.vgg16 import preprocess_input
import numpy as np
from keras.preprocessing.image import img_to_array

model = Model(inputs=model1.inputs, outputs=model1.layers[1].output)
image = load_img("/content/test/apples/img_p1_36.jpeg", target_size=(224,224))
image = img_to_array(image)
image = np.expand_dims(image, axis=0)
image = preprocess_input(image)
image

In [ ]:
features = model.predict(image)
fig = plt.figure(figsize=(20,15))
for i in range(1, features.shape[3]+1):
  plt.subplot(8, 8, i)
  plt.imshow(features[0, :, :, i-1], cmap='gray')

In [ ]:
model2 = VGG16()

layer_index = [ 2, 5 , 9 , 13 , 17]
outputs = [model2.layers[i].output for i in layer_index]
model2 = Model(inputs=model2.inputs, outputs=outputs)

In [ ]:
feature_map = model2.predict(image)

for i,fmap in zip(layer_index,feature_map):
    fig = plt.figure(figsize=(20,15))
    fig.suptitle("Layer_{}".format(i) , fontsize=20)
    for i in range(1,features.shape[3]+1):

        plt.subplot(8,8,i)
        plt.imshow(fmap[0,:,:,i-1] , cmap='gray')

plt.show()

In [ ]:
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img = image.load_img('/content/test/apples/img_p1_111.jpeg', target_size=(200, 200))
plt.imshow(img)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

In [ ]:
img = image.img_to_array(img)
print(img.shape)
input_batch = img.reshape(1, 200, 200, 3)

In [ ]:
i = 0
for output in datagen.flow(input_batch, batch_size=1, save_to_dir='aug'):
  i = i + 1
  if i == 10:
    break

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

aug_dir = 'aug'
augmented_files = [os.path.join(aug_dir, f) for f in os.listdir(aug_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

plt.figure(figsize=(15, 15))
for i, img_path in enumerate(augmented_files):
    if i >= 10:
        break
    plt.subplot(5, 2, i + 1)
    img = mpimg.imread(img_path)
    plt.imshow(img)
    plt.title(os.path.basename(img_path))
    plt.axis('off')
plt.tight_layout()
plt.show()